# Type II forecasting: the data

What the model is asked to do, before any model is involved:

> From one SDO frame at issue time *t* (00/06/12/18 UTC), give the probability that a
> **Type II burst** starts in *(t, t + 24 h]*, and that an **interplanetary-reaching** one
> (ending below 1 MHz) does.

This notebook looks at the labels, the splits and the data budget. It needs only pandas and
matplotlib, except the optional last section, which loads one SDO frame.

The files it reads are built by `python -m downstream_apps.radioburst.prepare_data`.

**It does not look at the test labels on purpose.** Test (2020–2024) is for the final score
only; every choice made here uses train and val.

In [ ]:
import os
import sys
from pathlib import Path

# Run from anywhere inside the repo: put the repo root on the path.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "workshop_infrastructure").is_dir())
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from downstream_apps.radioburst.configs import load_radioburst_config

cfg = load_radioburst_config("downstream_apps/radioburst/configs/config_script.yaml")

## The events

One row per decameter–hectometric Type II onset from the CDAW Wind/WAVES catalog, SDO era
only. `reaches_ip` marks bursts ending below 1 MHz: the shock survived into interplanetary
space. `behind_limb` marks source flares SDO cannot see.

In [ ]:
events = pd.read_csv(cfg.data.ds_events_path, parse_dates=["onset"])
print(f"{len(events)} onsets | reach IP: {events.reaches_ip.sum()} | behind the limb: {events.behind_limb.sum()}")

per_year = events.groupby([events.onset.dt.year, "reaches_ip"]).size().unstack(fill_value=0)
ax = per_year.rename(columns={0: "ends above 1 MHz", 1: "reaches IP (< 1 MHz)"}).plot.bar(stacked=True, figsize=(10, 3.5))
ax.axvline(per_year.index.get_loc(2020) - 0.5, color="k", ls="--")
ax.text(per_year.index.get_loc(2020) - 0.4, ax.get_ylim()[1] * 0.9, "test (cycle 25) →")
ax.set_xlabel(""); ax.set_ylabel("Type II onsets"); ax.set_title("Type II onsets per year")
plt.tight_layout()

Cycle 25 (2020–2024) produces about twice as many Type IIs per year as cycle 24. A model
trained on cycle 24 will therefore **under-forecast** probabilities on the test years, even if it
ranks well. TSS and ROC-AUC do not care; the Brier score and reliability curves do.

## The splits and their labels

- **train**: 4 of every 5 solar rotations of 2010–2019
- **val**: every 5th rotation of 2010–2019 (checkpoint selection, threshold choice)
- **test**: 2020–2024, never seen by Surya's pretraining

A 2-day gap at every boundary keeps label windows and near-identical frames from crossing
between splits.

In [ ]:
from downstream_apps.radioburst.evaluate import split_labels, rotation_of

labels = {s: split_labels(cfg, s, events) for s in ("train", "val")}
summary = pd.DataFrame({
    s: {"issue times": len(l), "P(Type II 24h)": l["type2"].mean(), "P(IP Type II 24h)": l["type2_ip"].mean(),
        "rotations": len(np.unique(rotation_of(l.index)))}
    for s, l in labels.items()
}).T
summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.2))
for y, (s, l) in enumerate(labels.items()):
    ax.plot(l.index, np.full(len(l), y), "|", ms=10, alpha=0.3, label=s)
ax.plot(events.onset, np.full(len(events), 2), "v", color="k", ms=4, label="Type II onset")
ax.set_yticks([0, 1, 2], ["train", "val", "onsets"]); ax.set_xlim(pd.Timestamp("2010-05"), pd.Timestamp("2020-01"))
ax.set_title("Issue times by split, 2010–2019 (test starts 2020)")
plt.tight_layout()

## How many independent events?

Events cluster: an active region can produce several Type IIs in a few days. The effective
sample size is closer to the number of **rotations containing events** than to the number of
events, which is why `evaluate.py` bootstraps over rotations.

In [ ]:
train_val = events[events.onset < pd.Timestamp("2020-01-01")]
per_rotation = pd.Series(rotation_of(train_val.onset)).value_counts()
print(f"{len(train_val)} onsets in 2010-2019, spread over {len(per_rotation)} rotations;"
      f" the busiest 5 rotations hold {per_rotation.head(5).sum()}")
per_rotation.value_counts().sort_index().plot.bar(figsize=(7, 3), title="Onsets per rotation (rotations with ≥1 onset)")
plt.xlabel("onsets in the rotation"); plt.ylabel("rotations"); plt.tight_layout()

## The data budget

Each SDO frame is about 1 GB and is cached locally on first read. Training on every train
issue time is ~11 TB; `data.ds_negative_ratio` keeps all positives and that many negatives per
positive. Validation and test are never subsampled (their scores must reflect the real
event rate).

In [ ]:
n_pos = int(labels["train"]["type2"].sum())
n_neg = len(labels["train"]) - n_pos
budget = pd.DataFrame(
    [{"ds_negative_ratio": r, "train frames": n_pos + min(n_neg, round(r * n_pos)) if r else n_pos + n_neg}
     for r in (1, 3, 10, None)]
)
budget["+ val frames"] = len(labels["val"])
budget["≈ TB total"] = ((budget["train frames"] + budget["+ val frames"]) / 1000).round(1)
budget

## How hard is it? The reference forecasts, on val

The model has to beat these. `mx_flare_24h` ("an M/X flare started in the past 24 h") is the
one that matters: it costs one line and already has real skill.

In [ ]:
from downstream_apps.radioburst.evaluate import reference_forecasts, tss

climatology = {n: float(labels["train"][n].mean()) for n in ("type2", "type2_ip")}
refs = reference_forecasts(cfg, labels["val"], events, climatology)
pd.DataFrame({
    name: {label: tss(labels["val"][label], refs[name][label]) for label in ("type2", "type2_ip")}
    for name in ("persistence", "mx_flare_24h")
}).T.rename_axis("TSS on val").round(3)

## Optional: one SDO frame

Needs the full `surya_ws` environment and an S3 cache directory (each frame ~1 GB). Set
`LOAD_FRAME = True` to run it.

In [ ]:
LOAD_FRAME = False

if LOAD_FRAME:
    from downstream_apps.radioburst.datasets.radioburst_dataset import TypeIIDataset
    from workshop_infrastructure.assets import ensure_assets
    from workshop_infrastructure.datasets.builders import _base_dataset_kwargs
    from workshop_infrastructure.utils import build_scalers

    ensure_assets(cfg, which=["scalers"])
    scalers = build_scalers(info=cfg.data.scalers_path)
    ds = TypeIIDataset(index_path=cfg.data.valid_data_path, phase="val", ds_events_path=cfg.data.ds_events_path,
                       ds_horizon=cfg.data.ds_horizon, **_base_dataset_kwargs(cfg, scalers))
    idx = int(np.flatnonzero(ds.labels["type2"].to_numpy())[0])  # first positive issue time in val
    sample = ds[idx]
    print(sample["ds_index"], "labels (type2, type2_ip):", sample["labels"])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, ch in zip(axes, ["aia193", "aia94", "hmi_m"]):
        ax.imshow(sample["ts"][cfg.data.channels.index(ch), 0], origin="lower", cmap="gray")
        ax.set_title(ch); ax.axis("off")
    plt.tight_layout()